# 04 · Pretrained convolutional network (transfer learning)

**ITRI626 · Colile Sibanda**

Objective **O4**. Runs on Colab; the laptop runs this notebook
only with `smoke.yaml`. The split is loaded from the committed files and
re-verified before a single batch is drawn.


In [ ]:
# On Colab only: mount Drive and install what the runtime is missing.
# from google.colab import drive; drive.mount('/content/drive')
# !pip -q install timm imagehash


In [ ]:
# Every notebook opens with the same four lines: path, config, seed, paths.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))   # run_status.py and run_claim.py at the root

from docclf.config.loader import load_config
from docclf.config.paths import resolve_paths
from docclf.config.seed import set_global_seed
from docclf.utils.env import detect_environment

CONFIG = ROOT / "configs" / "resnet50.yaml"
ENV = detect_environment()
cfg = load_config(CONFIG, env=ENV)
paths = resolve_paths(cfg, ENV).ensure()
set_global_seed(cfg.seed)

f"Config: {cfg.name} | Environment: {ENV} | Seed: {cfg.seed}"

## The transfer-learning entry

ImageNet weights, with the early stages frozen. MobileNetV3 is the lighter alternative to ResNet50; switching is a one-line config change, not a code change.


In [ ]:
from docclf.splits.io import load_splits
from docclf.splits.leakage import assert_no_group_overlap
from docclf.data.loaders import build_dataloaders

# `root=` resolves the split files' relative image paths against this
# machine, which is what lets the same committed split files work anywhere.
splits = load_splits(paths.splits, root=paths.root)
assert_no_group_overlap(splits, cfg.split.group_column)
loaders = build_dataloaders(splits, cfg)
{name: len(loader.dataset) for name, loader in loaders.items()}

In [ ]:
from docclf.models.registry import build_model, model_family
from docclf.models.summary import describe_model
from docclf.utils.env import select_device

model = build_model('resnet50', cfg.n_classes, cfg)
device = select_device(cfg)
summary = describe_model(model, 'resnet50')
print(f'Family: {model_family("resnet50")}')
print(f'Device: {device}')
summary.as_row()

In [ ]:
# Which layers are frozen, and what that costs in trainable parameters.
from docclf.models.summary import describe_model

print(f'freeze_upto: {cfg.train.freeze_upto}')
frozen = describe_model(model, 'resnet50')
print(f'Trainable parameters: {frozen.trainable_parameters:,}')
print(f'Frozen parameters: {frozen.frozen_parameters:,}')
print('Justification: the early stages encode generic edge and texture filters that\n'
      'transfer directly to document pages; the later stages encode object-level\n'
      'features that do not, and are fine-tuned on the page layouts instead.')

## Train

The loop is shared, unchanged, by all three models: whatever differs between them differs in the config, where it can be reported.


In [ ]:
# Training writes only into this model's own run directory, so it cannot
# overwrite another model's results, and a checkpoint is saved every epoch.
#
# Before starting, the model is claimed so that no second machine trains the
# same one. If a checkpoint is already there the run resumes from it, keeping
# the optimiser and learning-rate schedule, so a run paused on another machine
# is continued rather than restarted.
#
# This cell blocks for as long as training takes. For a run of several hours
# prefer `./train_model.sh resnet50` in a terminal, which survives the notebook
# being closed and evaluates the model when training finishes.
import subprocess

from docclf.training.trainer import fit
from docclf.training.class_weights import compute_class_weights
from run_status import progress, state_of, MODELS as RUN_SPECS

MODEL = "resnet50"
state = state_of(progress(MODEL, RUN_SPECS[MODEL]))
print(f"{MODEL} is {state}")
if state == "RUNNING":
    raise SystemExit(
        f"{MODEL} is being trained on another machine. Run `python run_status.py` "
        f"to see what is free."
    )

subprocess.run(
    [sys.executable, "run_claim.py", "--model", MODEL, "--note", "notebook"],
    cwd=ROOT, check=True,
)

weights = (
    compute_class_weights(splits[splits['split'] == 'train'], cfg.n_classes)
    if cfg.train.class_weights
    else None
)
resume = (paths.checkpoints / f"{MODEL}_last.pt").is_file()
if resume:
    print("Checkpoint found; resuming from the last completed epoch.")

try:
    history = fit(
        model, loaders, cfg, device, paths.checkpoints, weights,
        history_dir=paths.metrics, resume=resume,
    )
finally:
    subprocess.run(
        [sys.executable, "run_claim.py", "--model", MODEL, "--release"],
        cwd=ROOT, check=False,
    )

history_path = history.save(paths.metrics)
history.to_frame()


## Learning curves


In [ ]:
from docclf.evaluation.curves import plot_learning_curves
from IPython.display import Image as Show

figure_path = plot_learning_curves(history.to_frame(), paths.figures / 'curves_resnet50.png', 'resnet50')
best = history.best(cfg.train.monitor)
print(f'Best epoch: {best.epoch}')
print(f'Best validation macro-F1: {best.val_macro_f1:.4f}')
print(f'Mean seconds per epoch: {sum(r.seconds for r in history.records) / len(history.records):.1f}')
Show(str(figure_path))

## Evaluate on the held-out split

The one place this notebook touches the test split. It runs the same
`run_evaluate.py` the shell script does, so the files it writes are identical
whichever way the model was trained, and the comparison in notebook 06 can read
them without re-running inference.

This writes the test metrics, per-class scores, predictions, efficiency
measurement, error table, confusion matrices and learning curves into this
model's own run directory.

In [ ]:
# Evaluating is minutes, not hours: one pass over 2 400 pages plus the latency
# measurement. It loads the best checkpoint by validation macro-F1, never the
# last one, so the result does not depend on where training happened to stop.
import subprocess

result = subprocess.run(
    [
        sys.executable, "run_evaluate.py",
        "--config", "configs/resnet50.yaml",
        "--env", "local",
        "--model", "resnet50",
    ],
    cwd=ROOT,
)
if result.returncode != 0:
    raise SystemExit(
        "Evaluation failed. Train the model first, or run it in a terminal to "
        "see the full error."
    )

import pandas as pd

metrics = pd.read_csv(paths.metrics / "metrics_resnet50.csv")
per_class = pd.read_csv(paths.metrics / "per_class_resnet50.csv")
print(metrics.to_string(index=False))
print()
print(per_class.sort_values("f1").to_string(index=False))

## Record the run

This model is now complete: trained, evaluated, and its results written to its
own run directory. To fold it into the cross-model comparison:

```bash
python run_compare.py      # rebuilds every table and figure
python run_status.py       # confirms what is still outstanding
```

Then add the row to `docs/experiment-log.md`: date, config, commit, epochs,
seconds per epoch, best validation macro-F1, the held-out scores, and anything
that went wrong. A run that is not in the log did not happen.